# 5 · Build the server disk and installer ISO

## Goal

Construct a bootable server RAW disk, compress it, generate payload digests, build an installer UKI, and package the UEFI ISO. The tested builder runs these operations in one **atomic build transaction** to avoid mismatching server payloads and installer checksums.

## Prerequisites

The committed source tree must be clean, the public FreeIPA CA verified out-of-band, the Docker builder available, and `build/output` absent or empty. The build uses privileged Docker.

## Why the build stays atomic

The container entrypoint (`scripts/build-inside.sh`) now exposes **nine named functions**, each emitting an `SCOS_BUILD_STAGE` marker: verify toolchain → prepare workspace → validate/stage inputs → configure EFI signing → build server RAW → zstd compression → record payload digests → build installer UKI → assemble ISO → record evidence. These are individual, reviewable source blocks but execute as one transaction: the installer must always describe exactly the RAW built in the same run. The checks below inspect each output independently.

In [ ]:
from pathlib import Path
import subprocess, sys
START=Path.cwd().resolve()
ROOT=next((p for p in (START,*START.parents) if (p/"myst.yml").is_file() and (p/"scripts/build.sh").is_file()),None)
assert ROOT is not None, "Run from the SC Linux checkout (make book-lab)"
def run(*argv):
    print("+"," ".join(map(str,argv)),flush=True)
    return subprocess.run([str(a) for a in argv],cwd=ROOT,check=True)
print("Source:",ROOT)

## Action 1 · Check that source and output are safe to use

In [ ]:
status=subprocess.run(["git","status","--porcelain","--untracked-files=normal"],cwd=ROOT,capture_output=True,text=True,check=True).stdout
assert not status.strip(), f"Commit source changes before build:\n{status}"
OUTPUT=ROOT/"build/output"
assert not OUTPUT.exists() or not any(OUTPUT.iterdir()), "Choose a fresh output directory; current build/output is not empty"
assert (ROOT/"build/builder.id").is_file(), "Run chapter 4 first"
print("Image destination:",OUTPUT)

## Action 2 · Validate trust before the privileged build

This validates certificate structure and its checked-in digest. It cannot establish your independent verification.

In [ ]:
run("bash","scripts/notebook.sh","notebooks/check.ipynb","--require-trust")

## Action 3 · Construct both images

The builder stages the rootfs, runs `mkosi`, compresses the RAW, writes the payload manifest, builds the installer UKI, assembles the ISO and records provenance. The same step is used in CI.

In [ ]:
run("make","images")

## Action 4 · Check the server RAW and compressed disk

In [ ]:
raw=OUTPUT/"raw/sc-linux.raw"
compressed=OUTPUT/"sc-linux.raw.zst"
for path in (raw,compressed):
    assert path.is_file() and path.stat().st_size>1024*1024, f"Missing image: {path}"
    print(path.relative_to(OUTPUT), f"{path.stat().st_size:,} bytes")

## Action 5 · Check the payload manifest

The installer is bound to this specific compressed and uncompressed server image by SHA-256.

In [ ]:
import json
payload_path=OUTPUT/"payload.json"
payload=json.loads(payload_path.read_text())
assert set(payload)=={"raw","compressed"}
for key in ("raw","compressed"):
    assert payload[key]["size"]>0 and len(payload[key]["sha256"])==64
    print(key,payload[key]["sha256"][:16]+"…",payload[key]["size"])

## Action 6 · Check the installer UKI and ISO

The UKI boots the installer; the ISO includes the compressed server payload.

In [ ]:
for rel in ("installer/sc-linux-installer.efi","sc-linux.iso"):
    path=OUTPUT/rel
    assert path.is_file() and path.stat().st_size>1024*1024, f"Missing: {path}"
    print(rel, f"{path.stat().st_size:,} bytes")

## Action 7 · Confirm evidence was generated

In [ ]:
for rel in ("artifacts.json","footprint.json","SHA256SUMS","source-commit.txt"):
    path=OUTPUT/rel
    assert path.is_file() and path.stat().st_size>0, f"Missing evidence: {path}"
    print("Recorded:",rel)

## Check

The five required outputs exist and are non-empty. This **does not** establish that the RAW or ISO boots.

## Next

[Inspect the artifact checksums and footprint](06-artifacts.ipynb).